# Lab 11 – Statistical Data Analysis
**Big Data Techniques · "Gheorghe Asachi" Technical University of Iași**

This notebook covers all four exercises from Lab 11:

| # | Topic |
|---|-------|
| 1 | Uniform distribution – generate 10 000 integers in [1, 10] and compute empirical probabilities |
| 2 | Normal distribution – study how the sample mean converges to μ as n grows |
| 3 | Distribution fitting – empirical histogram + MLE for μ & σ + R² for dataset30 and dataset1000 |
| 4 | Monte Carlo – estimate π by sampling random points inside a unit square |

> All calculations are done from scratch (no scipy.stats) as required by the lab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

---
## Upload data files (Google Colab)
Run the cell below to upload `dataset30.txt` and `dataset1000.txt` from your computer.  
Skip this cell if the files are already in the same folder.

In [ ]:
# ── File upload for Google Colab ─────────────────────────────────────────────
try:
    from google.colab import files
    print('Running in Colab – please upload dataset30.txt and dataset1000.txt')
    uploaded = files.upload()   # a file-picker dialog will appear
except ImportError:
    print('Not in Colab – using files from the current directory')

---
## Exercise 1 – Uniform Distribution: Empirical Probabilities

A **uniform distribution** gives equal probability to every value in a range.  
For integers in [1, 10] the theoretical probability of each value is exactly **1/10 = 0.1**.

We generate 10 000 such integers and compare the *empirical* (observed) probabilities to the theoretical ones.

In [ ]:
# ── Generate 10 000 uniform integers in [1, 10] ───────────────────────────────
N_UNIFORM = 10_000
# np.random.randint(low, high) generates integers in [low, high)
# We want [1, 10] inclusive, so high = 11
uniformInts = np.random.randint(low=1, high=11, size=N_UNIFORM)

# ── Compute empirical probability for each unique value ───────────────────────
uniqueVals = np.arange(1, 11)          # values 1, 2, …, 10
# Count how many times each value appears, then divide by total to get probability
counts = np.array([(uniformInts == v).sum() for v in uniqueVals])
empiricalProbs = counts / N_UNIFORM    # P(X = v) ≈ count(v) / N

# ── Print results ────────────────────────────────────────────────────────────
print(f'Generated {N_UNIFORM} uniform integers in [1, 10]')
print(f'Theoretical probability for each value: {1/10:.4f}')
print()
print(f'{"Value":>6}  {"Count":>6}  {"Empirical P":>12}  {"Error vs 0.1":>13}')
print('-' * 45)
for v, c, p in zip(uniqueVals, counts, empiricalProbs):
    print(f'{v:>6}  {c:>6}  {p:>12.4f}  {abs(p - 0.1):>13.4f}')

# ── Bar chart: empirical vs theoretical ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(uniqueVals - 0.2, empiricalProbs, width=0.4,
       color='steelblue', label='Empirical probability')
# Horizontal reference line at the theoretical value 0.1
ax.axhline(y=0.1, color='red', linewidth=1.5, linestyle='--',
           label='Theoretical P = 0.1')
ax.set_xlabel('Value')
ax.set_ylabel('Probability')
ax.set_title(f'Exercise 1 – Uniform distribution: empirical probabilities (n={N_UNIFORM})')
ax.set_xticks(uniqueVals)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nWith large n the empirical probabilities converge to the theoretical 0.1.')
print(f'Max deviation observed: {np.max(np.abs(empiricalProbs - 0.1)):.4f}')

---
## Exercise 2 – Normal Distribution: Sample Mean Convergence

We generate samples of increasing size n = 100, 200, 300, …, 1000 from a normal distribution with **μ = 0, σ = 2**.

For each n we compute the **sample mean** and the **absolute error** `|sample_mean − μ|`.

This demonstrates the **Law of Large Numbers**: as n → ∞, the sample mean converges to the true mean μ.

In [ ]:
# ── Parameters of the normal distribution ────────────────────────────────────
MU    = 0    # true mean (μ)
SIGMA = 2    # true standard deviation (σ)

# Values of n to test
n_values = list(range(100, 1001, 100))   # [100, 200, 300, …, 1000]

sample_means = []
abs_errors   = []

print(f'Normal distribution: μ = {MU}, σ = {SIGMA}')
print(f'{"n":>5}  {"Sample mean":>12}  {"|mean - μ|":>12}')
print('-' * 35)

for n in n_values:
    # Generate n random values from the normal distribution
    # loc = mean (μ), scale = std deviation (σ)
    samples = np.random.normal(loc=MU, scale=SIGMA, size=n)

    # Compute the sample mean manually (sum / n)
    sample_mean = np.sum(samples) / n

    # Absolute error: how far the sample mean is from the true mean μ
    abs_error = abs(sample_mean - MU)

    sample_means.append(sample_mean)
    abs_errors.append(abs_error)
    print(f'{n:>5}  {sample_mean:>12.6f}  {abs_error:>12.6f}')

# ── Plot: sample mean and absolute error vs n ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: sample mean vs n
axes[0].plot(n_values, sample_means, 'o-', color='steelblue', label='Sample mean')
axes[0].axhline(MU, color='red', linestyle='--', linewidth=1.5, label=f'True μ = {MU}')
axes[0].set_xlabel('n')
axes[0].set_ylabel('Sample mean')
axes[0].set_title('Sample mean vs n')
axes[0].legend()
axes[0].grid(True)

# Right: absolute error vs n
axes[1].plot(n_values, abs_errors, 's-', color='darkorange', label='|mean - μ|')
axes[1].set_xlabel('n')
axes[1].set_ylabel('Absolute error')
axes[1].set_title('Absolute error |sample mean − μ| vs n')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Exercise 2 – Law of Large Numbers (μ=0, σ=2)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nObservation: as n increases, the absolute error generally decreases')
print('(though with random fluctuations) – the sample mean converges to μ.')

---
## Exercise 3 – Distribution Fitting with MLE + R²

**Steps:**
1. Read the dataset from file
2. Build the **empirical probability histogram** (count each unique value, divide by n)
3. Estimate **μ̂** and **σ̂** using **Maximum Likelihood Estimation** (MLE):
   - \$\hat{\mu} = \frac{\sum x_i}{n}\$ 
   - \$\hat{\sigma}^2 = \frac{\sum (x_i - \hat{\mu})^2}{n}\$
4. Evaluate the fit using the **coefficient of determination R²**:
   - \$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}\$
   - yᵢ = empirical probabilities, ŷᵢ = normal PDF at the same x values, ȳ = mean of yᵢ

In [ ]:
# ── Helper: read a dataset from a whitespace-separated text file ──────────────
def readDataset(fileName):
    """
    Read all whitespace-separated numbers from a file and return them
    as a NumPy array of floats.
    """
    with open(fileName, 'r') as f:
        data = f.read()
    return np.array(data.split()).astype(float)


# ── Helper: compute empirical probability histogram ───────────────────────────
def empiricalPDF(data):
    """
    Build the empirical (frequency) probability histogram.

    For each unique integer value v in data:
        P(X = v) = count(v) / n

    Returns
    -------
    uniqueVals : ndarray – sorted unique values in the dataset
    probs      : ndarray – empirical probability for each unique value
    """
    n = len(data)
    uniqueVals = np.sort(np.unique(data))   # sorted unique values
    counts = np.array([(data == v).sum() for v in uniqueVals])
    probs  = counts / n                      # normalise to probabilities
    return uniqueVals, probs


# ── Helper: MLE estimators for normal distribution ────────────────────────────
def mleFit(data):
    """
    Maximum Likelihood Estimation for a normal distribution.

    Eq. 2:  μ̂ = (1/n) * Σ xᵢ
    Eq. 3:  σ̂² = (1/n) * Σ (xᵢ − μ̂)²   →   σ̂ = sqrt(σ̂²)

    Returns estimated mean (mu_hat) and standard deviation (sigma_hat).
    """
    n      = len(data)
    mu_hat = np.sum(data) / n                          # Eq. 2
    var_hat = np.sum((data - mu_hat) ** 2) / n         # Eq. 3
    sigma_hat = math.sqrt(var_hat)
    return mu_hat, sigma_hat


# ── Helper: normal distribution PDF ──────────────────────────────────────────
def normalPDF(x, mu, sigma):
    """
    Evaluate the normal distribution PDF at value(s) x:
        F(x) = (1 / (σ√2π)) * exp(−(x − μ)² / (2σ²))    [Eq. 1]
    """
    return (1.0 / (sigma * math.sqrt(2 * math.pi))) * \
           np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))


# ── Helper: coefficient of determination R² ───────────────────────────────────
def computeR2(y_obs, y_est):
    """
    Coefficient of determination (Eq. 4):
        R² = 1 − Σ(yᵢ − ŷᵢ)² / Σ(yᵢ − ȳ)²

    y_obs : observed values (empirical probabilities)
    y_est : estimated values (normal PDF at the same x-points)

    R² ∈ (−∞, 1]; values closer to 1 indicate a better fit.
    """
    y_mean = np.mean(y_obs)
    ss_res = np.sum((y_obs - y_est) ** 2)   # residual sum of squares
    ss_tot = np.sum((y_obs - y_mean) ** 2)  # total sum of squares
    return 1.0 - ss_res / ss_tot


# ── Helper: full analysis for one dataset ─────────────────────────────────────
def analyseDataset(fileName, label):
    """
    Load a dataset, compute the empirical histogram, fit a normal distribution
    using MLE, compute R², and plot both distributions.
    """
    print(f'\n{"="*55}')
    print(f'Dataset: {label}  ({fileName})')
    print('='*55)

    # Step 1: load data
    data = readDataset(fileName)
    print(f'  n = {len(data)} values')
    print(f'  Min = {data.min():.0f},  Max = {data.max():.0f}')

    # Step 2: empirical probability histogram
    uniqueVals, empiricalProbs = empiricalPDF(data)

    # Step 3: MLE to find best-fit μ and σ
    mu_hat, sigma_hat = mleFit(data)
    print(f'\n  MLE estimates:')
    print(f'    μ̂ (mean)             = {mu_hat:.4f}')
    print(f'    σ̂ (std deviation)    = {sigma_hat:.4f}')

    # Step 4: evaluate the normal PDF at each unique value
    # These are the "estimated" probabilities ŷᵢ
    fittedProbs = normalPDF(uniqueVals, mu_hat, sigma_hat)

    # Step 5: compute R²
    r2 = computeR2(empiricalProbs, fittedProbs)
    print(f'\n  Coefficient of determination R² = {r2:.6f}')
    if r2 > 0.9:
        quality = 'excellent fit'
    elif r2 > 0.7:
        quality = 'good fit'
    elif r2 > 0.5:
        quality = 'moderate fit'
    else:
        quality = 'poor fit (more data needed, or not normally distributed)'
    print(f'  → {quality}')

    # Step 6: plot
    # Use a finer x-grid for the smooth continuous normal curve
    x_fine = np.linspace(data.min() - 1, data.max() + 1, 500)
    y_fine = normalPDF(x_fine, mu_hat, sigma_hat)

    fig, ax = plt.subplots(figsize=(9, 4))

    # Empirical histogram as blue bars (each bar width = 1)
    ax.bar(uniqueVals, empiricalProbs, width=0.8, color='steelblue',
           alpha=0.7, label='Empirical P(X = v)')

    # Fitted normal PDF as a smooth red curve
    ax.plot(x_fine, y_fine, color='red', linewidth=2,
            label=f'Normal PDF  μ̂={mu_hat:.2f}, σ̂={sigma_hat:.2f}')

    ax.set_xlabel('Value')
    ax.set_ylabel('Probability / PDF')
    ax.set_title(f'Exercise 3 – {label}: empirical vs fitted normal  (R²={r2:.4f})')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return mu_hat, sigma_hat, r2

In [ ]:
# ── Run analysis on both datasets ─────────────────────────────────────────────
mu30,   sigma30,   r2_30   = analyseDataset('dataset30.txt',   'dataset30  (n=30)')
mu1000, sigma1000, r2_1000 = analyseDataset('dataset1000.txt', 'dataset1000 (n=1000)')

# ── Side-by-side R² comparison ────────────────────────────────────────────────
print('\n' + '='*45)
print('Comparison of R² values:')
print(f'  dataset30   (n=30):    R² = {r2_30:.6f}')
print(f'  dataset1000 (n=1000):  R² = {r2_1000:.6f}')
print('='*45)
print('The larger dataset produces a higher R² because')
print('more data points reduce statistical noise.')

---
## Exercise 4 – Monte Carlo Estimation of π

**Idea:** inscribe a quarter-circle of radius R = 1 inside a unit square [0,1]×[0,1].

- Area of the square: \$S_1 = R^2 = 1\$
- Area of the quarter-circle: \$S_2 = \frac{\pi R^2}{4} = \frac{\pi}{4}\$
- Therefore: \$\pi = 4 \cdot \frac{S_2}{S_1}\$

**Simulation:** throw random points uniformly in the unit square.  
A point (x, y) is inside the quarter-circle if \$x^2 + y^2 \leq 1\$.  
The fraction of points inside the arc approximates S₂/S₁, so **π ≈ 4 × (points inside) / (total points)**.

In [ ]:
# ── Monte Carlo π estimation ──────────────────────────────────────────────────
N_SIMULATIONS = 30_000   # total number of random points to throw

np.random.seed(42)   # fix seed for reproducibility (remove to get a different result each time)

# Generate all N random (x, y) points at once – much faster than a Python loop
x_rand = np.random.uniform(0, 1, N_SIMULATIONS)   # x-coordinates in [0, 1]
y_rand = np.random.uniform(0, 1, N_SIMULATIONS)   # y-coordinates in [0, 1]

# A point is inside the quarter-circle if x² + y² ≤ 1 (radius R = 1)
inside = (x_rand ** 2 + y_rand ** 2) <= 1.0   # boolean array

# Running estimate of π after each simulation:
# at step i: S1 = i+1,  S2 = number of points inside so far
S2_running = np.cumsum(inside)              # cumulative count of points inside
S1_running = np.arange(1, N_SIMULATIONS + 1)  # total points seen so far
pi_estimates = 4.0 * S2_running / S1_running  # running π estimate

# Final estimate
pi_final = pi_estimates[-1]
print(f'Monte Carlo π estimate after {N_SIMULATIONS} simulations: {pi_final:.6f}')
print(f'True π:                                               {math.pi:.6f}')
print(f'Absolute error:                                       {abs(pi_final - math.pi):.6f}')

In [ ]:
# ── Plot 1: convergence of π estimate over simulations ───────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(S1_running, pi_estimates, color='steelblue', linewidth=0.8,
        label='π estimate')
ax.axhline(math.pi, color='red', linestyle='--', linewidth=1.5,
           label=f'True π = {math.pi:.6f}')
ax.set_xlabel('Number of simulations')
ax.set_ylabel('π estimate')
ax.set_title('Exercise 4 – Monte Carlo π convergence')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: scatter of sampled points coloured by inside/outside ──────────────
# Only visualise a subset to keep the plot readable
N_PLOT = min(N_SIMULATIONS, 5_000)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')

# Points outside the quarter-circle (red)
ax.scatter(x_rand[:N_PLOT][~inside[:N_PLOT]],
           y_rand[:N_PLOT][~inside[:N_PLOT]],
           s=1, color='red', alpha=0.5, label='Outside arc')

# Points inside the quarter-circle (blue)
ax.scatter(x_rand[:N_PLOT][inside[:N_PLOT]],
           y_rand[:N_PLOT][inside[:N_PLOT]],
           s=1, color='steelblue', alpha=0.5, label='Inside arc')

# Draw the arc boundary: x² + y² = 1  →  y = sqrt(1 - x²)
arc_x = np.linspace(0, 1, 500)
arc_y = np.sqrt(1 - arc_x ** 2)
ax.plot(arc_x, arc_y, color='black', linewidth=1.5, label='Quarter-circle boundary')

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Monte Carlo: n={N_PLOT},  π ≈ {4 * inside[:N_PLOT].sum() / N_PLOT:.4f}')
ax.legend(markerscale=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: six snapshots at increasing n (replicates Fig. 6 from the lab PDF) ─
snapshot_ns = [3000, 5000, 8500, 15000, 24000, 30000]
snapshot_ns = [n for n in snapshot_ns if n <= N_SIMULATIONS]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for ax, snap_n in zip(axes.flat, snapshot_ns):
    ax.set_aspect('equal')

    xs = x_rand[:snap_n]
    ys = y_rand[:snap_n]
    ins = inside[:snap_n]
    pi_snap = 4.0 * ins.sum() / snap_n

    ax.scatter(xs[~ins], ys[~ins], s=0.5, color='red',  alpha=0.5)
    ax.scatter(xs[ins],  ys[ins],  s=0.5, color='steelblue', alpha=0.5)
    ax.plot(arc_x, arc_y, color='black', linewidth=1)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.set_title(f'n={snap_n:,},  π≈{pi_snap:.4f}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Exercise 4 – Monte Carlo snapshots (blue = inside arc)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

| Exercise | Key result |
|----------|------------|
| 1 – Uniform | With 10 000 samples each value in [1,10] has empirical probability ≈ 0.1 (theoretical = 0.1 exactly) |
| 2 – Normal mean | As n grows from 100 to 1000 the absolute error `|mean − μ|` generally decreases — Law of Large Numbers |
| 3 – MLE fit | dataset30 (n=30) has a lower R² than dataset1000 (n=1000): more data → better fit |
| 4 – Monte Carlo | With 30 000 simulations π is estimated to within ~0.001 of the true value; more points → closer estimate |